# Experimental feature-task MoE/MTL

This is a deliberately small experiment for `optimal_trader`. Each scalar feature in one FMP technical feature family is treated as an MTL task. Every task predicts the same next-session return, and a learned task-to-expert gate is inspected as a candidate feature-family discovery mechanism.

The universe is screened at `min_market_cap = $1T`. Data comes through Quant Warehouse; this notebook does not call FMP directly. The experiment uses every feature from all six curated FMP technical families.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'optimal_trader').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
WORKSPACE_ROOT = REPO_ROOT
for repo in ('quant-warehouse', 'quant-orchestrator'):
    path = WORKSPACE_ROOT / repo
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from quant_warehouse import Warehouse
from quant_warehouse.research_tools import (
    FamilyEvaluationConfig,
    build_fundamental_feature_panel,
    build_technical_feature_panel,
    screen_fmp_equity_universe,
)

SEED = 20260713
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '2018-01-01'
TRAIN_END = '2023-12-31'
VALID_END = '2024-12-31'
TARGET_FAMILY = 'fmp.technical_momentum'
TARGET_FEATURE_COUNT = 50
N_EXPERTS = None  # filled as ceil(sqrt(number_of_tasks))
FEATURE_WORKERS = 12
BATCH_SIZE = 16384
USE_AMP = True
TARGET_HORIZONS = {'1d': 1, '5d': 5, '1m': 21, '6m': 126, '1y': 252}
TARGET_LABELS = ('1d', '5d', '1m', '6m', 'ytd', '1y')
TARGET_COLUMNS = [f'forward_return_{label}' for label in TARGET_LABELS]

print({'repo_root': str(REPO_ROOT), 'device': str(DEVICE), 'torch': torch.__version__})

{'repo_root': '/home/jlee153232/PycharmProjects', 'device': 'cpu', 'torch': '2.10.0+cpu'}


## Screen the $1T universe and build all technical families


In [2]:
wh = Warehouse()
config = FamilyEvaluationConfig(
    provider='fmp', market_cap_min=MIN_MARKET_CAP, country='US',
    exchanges=('NASDAQ', 'NYSE', 'AMEX'), start_date=START_DATE,
)
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(config, warehouse=wh)
print({'universe_source': universe_source, 'eligible_symbols': len(symbols)})

# Build all six curated families so every generated column becomes a task.
technical_sources = tuple(f'fmp.{name}' for name in (
    'technical_candles', 'technical_cycles', 'technical_math',
    'technical_momentum', 'technical_overlap', 'technical_performance',
))
feature_panel, feature_metadata, diagnostics, timings = build_technical_feature_panel(
    symbols, config, strategy_sources=technical_sources, warehouse=wh, max_workers=FEATURE_WORKERS
)
fundamental_panel, fundamental_metadata, fundamental_diagnostics, fundamental_timings = build_fundamental_feature_panel(
    symbols, config, warehouse=wh
)
fundamental_features = fundamental_metadata['feature'].drop_duplicates().tolist()
technical_features = feature_metadata['feature'].drop_duplicates().tolist()
feature_panel = fundamental_panel[['symbol', 'date', *fundamental_features]].merge(
    feature_panel[['symbol', 'date', *technical_features]], on=['symbol', 'date'], how='inner'
)
feature_metadata = pd.concat([fundamental_metadata, feature_metadata], ignore_index=True, sort=False).drop_duplicates()
family_counts = (feature_metadata.groupby(['source', 'family'])['feature']
                 .nunique().sort_values(ascending=False).rename('feature_count').reset_index())
family_counts['distance_from_50'] = (family_counts['feature_count'] - TARGET_FEATURE_COUNT).abs()
display(family_counts.sort_values(['distance_from_50', 'family']))

print({'feature_families': [f'{row.source}.{row.family}' for row in family_counts.itertuples()], 'total_features': int(family_counts['feature_count'].sum())})

{'universe_source': 'catalog:fmp:fallback_after_OpenBBError+openbb:fmp', 'eligible_symbols': 14}


,source,family,feature_count,distance_from_50
1,fmp,technical_momentum,42,8
2,fmp,technical_overlap,35,15
0,fmp,technical_candles,72,22
3,fmp,technical_math,21,29
4,fmp,technical_performance,9,41
5,fmp,technical_cycles,8,42


{'selected_family': 'fmp.technical_momentum', 'feature_count': 42, 'distance_from_50': 8}


In [3]:
features = feature_metadata['feature'].drop_duplicates().sort_values().tolist()
feature_train_coverage = feature_panel[features].notna().sum().astype(int)
excluded_no_train_coverage = feature_train_coverage.loc[feature_train_coverage.eq(0)].index.tolist()
features = [feature for feature in features if feature not in set(excluded_no_train_coverage)]
print({'excluded_zero_train_coverage': len(excluded_no_train_coverage), 'remaining_tasks': len(features)})
panel = feature_panel[['symbol', 'date', *features]].copy()
panel['date'] = pd.to_datetime(panel['date']).dt.normalize()
print({'tasks': len(features), 'experts': int(N_EXPERTS or np.ceil(np.sqrt(len(features)))), 'rows': len(panel)})
display(pd.DataFrame({'task_id': np.arange(len(features)), 'feature': features}))

{'tasks': 42, 'experts': 7, 'rows': 27855}


,task_id,feature
0,0,ta_momentum__adosc_3_10
1,1,ta_momentum__adx_14
2,2,ta_momentum__adx_14_dmn_14
3,3,ta_momentum__adx_14_dmp_14
4,4,ta_momentum__ao_5_34
5,5,ta_momentum__atr_14_at_rr_14
6,6,ta_momentum__bop
7,7,ta_momentum__cci_20_0_015
8,8,ta_momentum__cmo_14
9,9,ta_momentum__dpo_20


## Add the one-day forward-return target

The target is formed from the next available warehouse close for each symbol. Rows without a next session are removed. Standardization statistics are fit on training rows only.

In [4]:
price_parts = []
for symbol in symbols:
    prices = wh.read_prices(symbol, provider='fmp', start=START_DATE)
    if prices is None or prices.empty:
        continue
    p = prices.rename(columns=str.lower).reset_index()
    if 'date' not in p.columns:
        p = p.rename(columns={p.columns[0]: 'date'})
    if 'close' not in p.columns:
        continue
    p['symbol'] = symbol
    p['date'] = pd.to_datetime(p['date']).dt.normalize()
    close = pd.to_numeric(p['close'], errors='coerce')
    for label, sessions in TARGET_HORIZONS.items():
        p[f'forward_return_{label}'] = close.groupby(p['symbol']).shift(-sessions) / close - 1.0
    year_end_close = close.groupby([p['symbol'], p['date'].dt.year]).transform('last')
    p['forward_return_ytd'] = year_end_close / close - 1.0
    price_parts.append(p[['symbol', 'date', *TARGET_COLUMNS]])
targets = pd.concat(price_parts, ignore_index=True)
dataset = panel.merge(targets, on=['symbol', 'date'], how='inner')
dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna(subset=TARGET_COLUMNS).sort_values(['date', 'symbol'])
dataset = dataset.dropna(subset=features, how='all').reset_index(drop=True)
print({'dataset_rows': len(dataset), 'symbols': dataset['symbol'].nunique(), 'date_min': dataset['date'].min(), 'date_max': dataset['date'].max()})
display(dataset[features + TARGET_COLUMNS].describe().T[['count', 'mean', 'std', 'min', 'max']].tail(len(TARGET_COLUMNS) + 3))

{'dataset_rows': 27841, 'symbols': 14, 'date_min': Timestamp('2018-01-02 00:00:00'), 'date_max': Timestamp('2026-07-09 00:00:00')}


,count,mean,std,min,max
ta_momentum__adosc_3_10,27841.0,1.078222e+07,8.110379e+07,-9.476954e+08,1.004599e+09
ta_momentum__adx_14,27841.0,2.456722e+01,1.018950e+01,0.000000e+00,8.955797e+01
ta_momentum__adx_14_dmn_14,27841.0,2.233273e+01,7.671506e+00,0.000000e+00,5.821209e+01
ta_momentum__adx_14_dmp_14,27841.0,2.586790e+01,8.448532e+00,0.000000e+00,7.563210e+01
ta_momentum__ao_5_34,27820.0,2.262184e+02,3.967628e+03,-5.423722e+04,4.646768e+04
ta_momentum__atr_14_at_rr_14,27841.0,5.991499e+02,2.241984e+03,0.000000e+00,2.542136e+04
ta_momentum__bop,27841.0,2.476024e-02,5.309583e-01,-1.000000e+00,1.000000e+00
ta_momentum__cci_20_0_015,27841.0,2.207527e+01,1.094169e+02,-4.216294e+02,5.039578e+02
ta_momentum__cmo_14,27841.0,8.278106e+00,3.349681e+01,-9.880418e+01,1.000000e+02
ta_momentum__dpo_20,27820.0,3.193866e+02,5.743195e+03,-9.043600e+04,7.245568e+04


## Feature-task MoE/MTL model

Each task receives exactly one scalar feature. A task embedding produces a task-specific gate over experts. Each expert processes the scalar plus that embedding. Every task has an auxiliary return head; the mean of task predictions is the primary prediction. The gate is therefore interpretable as a learned feature-to-expert assignment.

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

class FeatureTaskMoE(nn.Module):
    def __init__(self, n_tasks, n_experts, n_outputs=2, embedding_dim=12, hidden_dim=32):
        super().__init__()
        self.n_tasks, self.n_experts, self.n_outputs = n_tasks, n_experts, n_outputs
        self.task_embedding = nn.Embedding(n_tasks, embedding_dim)
        self.gate = nn.Linear(embedding_dim, n_experts)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(1 + embedding_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim), nn.GELU()) for _ in range(n_experts)])
        self.task_head_weight = nn.Parameter(torch.empty(n_tasks, hidden_dim, n_outputs))
        self.task_head_bias = nn.Parameter(torch.zeros(n_tasks, n_outputs))
        nn.init.xavier_uniform_(self.task_head_weight)

    def forward(self, x, coverage=None):  # x: [batch, task, 1], coverage: [batch, task]
        b = x.shape[0]; emb = self.task_embedding.weight
        gate = self.gate(emb).softmax(-1)
        task_emb = emb.unsqueeze(0).expand(b, -1, -1)
        h = torch.cat([x, task_emb], dim=2)  # [batch, task, scalar + embedding]
        expert_h = torch.stack([expert(h) for expert in self.experts], dim=2)  # [batch, task, expert, hidden]
        mixed = (expert_h * gate.unsqueeze(0).unsqueeze(-1)).sum(dim=2)
        task_predictions = torch.einsum('bth,tho->bto', mixed, self.task_head_weight) + self.task_head_bias.unsqueeze(0)
        if coverage is None:
            aggregate = task_predictions.mean(dim=1)
        else:
            weights = coverage.to(task_predictions.dtype).unsqueeze(-1)
            aggregate = (task_predictions * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)
        return aggregate, task_predictions, gate

def make_split(frame):
    train = frame['date'].le(pd.Timestamp(TRAIN_END))
    valid = frame['date'].gt(pd.Timestamp(TRAIN_END)) & frame['date'].le(pd.Timestamp(VALID_END))
    test = frame['date'].gt(pd.Timestamp(VALID_END))
    return train, valid, test

train_mask, valid_mask, test_mask = make_split(dataset)
mu = dataset.loc[train_mask, features].median().fillna(0)
sigma = dataset.loc[train_mask, features].std().replace(0, 1).fillna(1)
feature_coverage = dataset[features].notna().to_numpy('float32')
X = ((dataset[features].fillna(mu) - mu) / sigma).clip(-8, 8).to_numpy('float32')[:, :, None]
target_mu = dataset.loc[train_mask, TARGET_COLUMNS].mean()
target_sigma = dataset.loc[train_mask, TARGET_COLUMNS].std().replace(0, 1).fillna(1)
y = ((dataset[TARGET_COLUMNS] - target_mu) / target_sigma).to_numpy('float32')
# The prepared experiment tensors are small enough to keep resident on the CUDA device.
X_device = torch.from_numpy(X).to(DEVICE)
coverage_device = torch.from_numpy(feature_coverage).to(DEVICE)
y_device = torch.from_numpy(y).to(DEVICE)
coverage_counts = pd.DataFrame({'feature': features, 'coverage_rows': feature_coverage.sum(axis=0).astype(int), 'train_coverage_rows': feature_coverage[train_mask].sum(axis=0).astype(int)}).sort_values('coverage_rows')
display(coverage_counts.head(20))

def loader(mask, shuffle=False):
    indices = torch.from_numpy(np.flatnonzero(mask)).to(DEVICE)
    if shuffle:
        indices = indices[torch.randperm(indices.numel(), device=DEVICE)]
    for start in range(0, indices.numel(), BATCH_SIZE):
        batch_indices = indices[start:start + BATCH_SIZE]
        yield X_device[batch_indices], y_device[batch_indices], coverage_device[batch_indices]

n_experts = int(N_EXPERTS or np.ceil(np.sqrt(len(features))))
model = FeatureTaskMoE(len(features), n_experts, n_outputs=len(TARGET_LABELS)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss()
amp_enabled = bool(USE_AMP and DEVICE.type == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
history = []
for epoch in range(40):
    model.train(); train_losses = []
    for xb, yb, mb in loader(train_mask, shuffle=True):
        xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE); optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
            aggregate, task_pred, gate = model(xb, mb)
            task_loss = nn.functional.smooth_l1_loss(task_pred, yb[:, None, :].expand_as(task_pred), reduction='none')
            task_loss = (task_loss * mb[:, :, None]).sum() / mb.sum().clamp_min(1.0)
            loss = loss_fn(aggregate, yb) + 0.25 * task_loss
            gate_entropy = -(gate * gate.clamp_min(1e-8).log()).sum()
            load_balance = (gate.mean(dim=0) - (1.0 / n_experts)).pow(2).sum()
            loss = loss + 0.001 * gate_entropy + 0.01 * load_balance
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update(); train_losses.append(loss.item())
    model.eval(); valid_losses = []
    with torch.no_grad():
        for xb, yb, mb in loader(valid_mask):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
                aggregate, task_pred, _ = model(xb, mb); valid_losses.append(loss_fn(aggregate, yb).item())
    history.append({'epoch': epoch + 1, 'train_loss': np.mean(train_losses), 'valid_loss': np.mean(valid_losses) if valid_losses else np.nan})
display(pd.DataFrame(history).tail())

## Evaluate and inspect learned feature families

The routing table is the main output. Features with similar gate distributions are candidate learned families. Stability should be tested by repeating this notebook across seeds and rolling time splits before treating the groups as meaningful.

In [ ]:
def predict(mask):
    model.eval(); preds, actual = [], []
    with torch.no_grad():
        for xb, yb, mb in loader(mask):
            pred, _, _ = model(xb, mb); preds.extend(pred.float().cpu().numpy()); actual.extend(yb.float().cpu().numpy())
    actual = np.asarray(actual) * target_sigma.to_numpy() + target_mu.to_numpy()
    preds = np.asarray(preds) * target_sigma.to_numpy() + target_mu.to_numpy()
    return actual, preds

for split, mask in [('train', train_mask), ('valid', valid_mask), ('test', test_mask)]:
    actual, pred = predict(mask)
    metrics = {'split': split, 'rows': len(actual)}
    for horizon, label in enumerate(TARGET_LABELS):
        metrics[f'{label}_rmse'] = float(np.sqrt(np.mean((pred[:, horizon] - actual[:, horizon]) ** 2))) if len(actual) else np.nan
        metrics[f'{label}_directional_accuracy'] = float(np.mean(np.sign(pred[:, horizon]) == np.sign(actual[:, horizon]))) if len(actual) else np.nan
        metrics[f'{label}_pearson'] = float(np.corrcoef(actual[:, horizon], pred[:, horizon])[0, 1]) if len(actual) > 2 else np.nan
    print(metrics)

with torch.no_grad():
    _, _, gate = model(torch.zeros((1, len(features), 1), device=DEVICE))
routing = pd.DataFrame(gate.cpu().numpy(), index=features, columns=[f'expert_{i}' for i in range(n_experts)])
routing['assigned_expert'] = routing.idxmax(axis=1)
routing['assignment_confidence'] = routing.drop(columns='assigned_expert').max(axis=1)
display(routing.sort_values(['assigned_expert', 'assignment_confidence'], ascending=[True, False]))
display(routing.groupby('assigned_expert').size().rename('feature_count').to_frame())
print('\nComplete expert assignments:')
for expert, group in routing.groupby('assigned_expert', sort=True):
    print(f'{expert} ({len(group)}): ' + ', '.join(group.index.tolist()))